In [ ]:
# 1. Установка и авторизация Kaggle
!pip install -q kaggle

import os, zipfile, shutil
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Загрузите kaggle.json")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    for fname in uploaded.keys():
        if fname == 'kaggle.json':
            shutil.move(fname, '/root/.kaggle/kaggle.json')
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
            break


Загрузите kaggle.json


Saving kaggle.json to kaggle.json


In [ ]:
# 2. Скачивание и распаковка полного датасета в /content

!kaggle competitions download -c dogs-vs-cats-redux-kernels-edition -p /content/data
!yes | unzip -q /content/data/dogs-vs-cats-redux-kernels-edition.zip -d /content/data
!yes | unzip -q /content/data/train.zip -d /content/data
!yes | unzip -q /content/data/test.zip -d /content/data
!rm -f /content/data/*.zip

TRAIN_DIR = '/content/data/train'
TEST_DIR = '/content/data/test'
print("Train files:", len(os.listdir(TRAIN_DIR)))
print("Test files:", len(os.listdir(TEST_DIR)))

100% 814M/814M [00:09<00:00, 89.0MB/s]

Train files: 25000
Test files: 12500


In [ ]:
# Проверка

import os

TRAIN_DIR = '/content/data/train'
TEST_DIR  = '/content/data/test'

print("Папка train существует:", os.path.exists(TRAIN_DIR))
print("Папка test существует:", os.path.exists(TEST_DIR))

if os.path.exists(TRAIN_DIR):
    train_files = os.listdir(TRAIN_DIR)
    print(f"Количество файлов в train: {len(train_files)}")
    if len(train_files) > 0:
        print("Примеры файлов:", train_files[:5])

if os.path.exists(TEST_DIR):
    test_files = os.listdir(TEST_DIR)
    print(f"Количество файлов в test: {len(test_files)}")
    if len(test_files) > 0:
        print("Примеры файлов:", test_files[:5])

# Проверка, что архивы удалены
zip_files = [f for f in os.listdir('/content/data') if f.endswith('.zip')]
print(f"Осталось zip-архивов: {len(zip_files)} (должно быть 0)")

Папка train существует: True
Папка test существует: True
Количество файлов в train: 25000
Примеры файлов: ['dog.3184.jpg', 'dog.1430.jpg', 'cat.6767.jpg', 'cat.2940.jpg', 'dog.1852.jpg']
Количество файлов в test: 12500
Примеры файлов: ['8172.jpg', '10342.jpg', '9151.jpg', '11231.jpg', '7460.jpg']
Осталось zip-архивов: 0 (должно быть 0)


In [ ]:
# 3. Импорты, параметры, tf.data
import re, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split

tf.keras.mixed_precision.set_global_policy('mixed_float16')

IMG_SIZE = (160, 160)
BATCH_SIZE = 128
EPOCHS1, EPOCHS2 = 12, 8
SEED = 42

def get_files_labels(dir_path):
    files, labels = [], []
    for fname in os.listdir(dir_path):
        if fname.endswith('.jpg'):
            files.append(os.path.join(dir_path, fname))
            labels.append(1.0 if fname.startswith('dog') else 0.0)
    return files, labels

train_files, train_labels = get_files_labels(TRAIN_DIR)
test_files = sorted([os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR) if f.endswith('.jpg')])

train_files, val_files, train_labels, val_labels = train_test_split(
    train_files, train_labels, test_size=0.2, random_state=SEED, stratify=train_labels
)

def decode_image(fpath, label):
    img = tf.io.read_file(fpath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img, label

def create_ds(files, labels, batch_size, shuffle=True, repeat=False):
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    if shuffle:
        ds = ds.shuffle(len(files), seed=SEED)
    ds = ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if repeat:
        ds = ds.repeat()
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_ds(train_files, train_labels, BATCH_SIZE, shuffle=True, repeat=True)
val_ds   = create_ds(val_files,   val_labels,   BATCH_SIZE, shuffle=False, repeat=False)

steps_per_epoch = len(train_files) // BATCH_SIZE
val_steps = max(1, len(val_files) // BATCH_SIZE)

In [ ]:
# 4. Модель
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base.trainable = False
x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.5)(x)
out = layers.Dense(1, activation='sigmoid', dtype='float32')(x)
model = models.Model(inputs=base.input, outputs=out)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.2, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best.h5', save_best_only=True, verbose=1)
]

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# 5. Этап обучения (голова + тонкая настройка)

model.fit(train_ds, steps_per_epoch=steps_per_epoch, epochs=EPOCHS1,
          validation_data=val_ds, validation_steps=val_steps, callbacks=callbacks, verbose=1)

base.trainable = True
for layer in base.layers[:100]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_ds, steps_per_epoch=steps_per_epoch, epochs=EPOCHS2,
          validation_data=val_ds, validation_steps=val_steps, callbacks=callbacks, verbose=1)

model.load_weights('best.h5')

Epoch 1/12
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step - accuracy: 0.9529 - loss: 0.1548
Epoch 1: val_loss improved from None to 0.08667, saving model to best.h5



Epoch 1: finished saving model to best.h5
156/156 ━━━━━━━━━━━━━━━━━━━━ 103s 364ms/step - accuracy: 0.9747 - loss: 0.1109 - val_accuracy: 0.9816 - val_loss: 0.0867 - learning_rate: 0.0010
Epoch 2/12
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9835 - loss: 0.0811
Epoch 2: val_loss improved from 0.08667 to 0.08064, saving model to best.h5



Epoch 2: finished saving model to best.h5
156/156 ━━━━━━━━━━━━━━━━━━━━ 10s 61ms/step - accuracy: 0.9840 - loss: 0.0790 - val_accuracy: 0.9824 - val_loss: 0.0806 - learning_rate: 0.0010
Epoch 3/12
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9872 - loss: 0.0679
Epoch 3: val_loss did not improve from 0.08064
156/156 ━━━━━━━━━━━━━━━━━━━━ 10s 63ms/step - accuracy: 0.9868 - loss: 0.0675 - val_accuracy: 0.9822 - val_loss: 0.0816 - learning_rate: 0.0010
Epoch 4/12
155/156 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9892 - loss: 0.0618
Epoch 4: val_loss did not improve from 0.08064
156/156 ━━━━━━━━━━━━━━━━━━━━ 8s 54ms/step - accuracy: 0.9895 - loss: 0.0611 - val_accuracy: 0.9820 - val_loss: 0.0807 - learning_rate: 0.0010
Epoch 5/12
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9881 - loss: 0.0603
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.

Epoch 5: val_loss did not improve from 0.08064
156/156 ━━━━━━━━━━━━━━━━━━━━ 9s 58ms/step - accur


Epoch 6: finished saving model to best.h5
156/156 ━━━━━━━━━━━━━━━━━━━━ 9s 61ms/step - accuracy: 0.9935 - loss: 0.0478 - val_accuracy: 0.9820 - val_loss: 0.0740 - learning_rate: 2.0000e-04
Epoch 7/12
155/156 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9947 - loss: 0.0445
Epoch 7: val_loss improved from 0.07401 to 0.07370, saving model to best.h5



Epoch 7: finished saving model to best.h5
156/156 ━━━━━━━━━━━━━━━━━━━━ 10s 67ms/step - accuracy: 0.9950 - loss: 0.0439 - val_accuracy: 0.9824 - val_loss: 0.0737 - learning_rate: 2.0000e-04
Epoch 8/12
155/156 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.9949 - loss: 0.0423
Epoch 8: val_loss did not improve from 0.07370
156/156 ━━━━━━━━━━━━━━━━━━━━ 9s 58ms/step - accuracy: 0.9951 - loss: 0.0413 - val_accuracy: 0.9814 - val_loss: 0.0748 - learning_rate: 2.0000e-04
Epoch 9/12
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9956 - loss: 0.0407
Epoch 9: val_loss did not improve from 0.07370
156/156 ━━━━━━━━━━━━━━━━━━━━ 9s 58ms/step - accuracy: 0.9957 - loss: 0.0395 - val_accuracy: 0.9822 - val_loss: 0.0737 - learning_rate: 2.0000e-04
Epoch 10/12
155/156 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9963 - loss: 0.0382
Epoch 10: val_loss improved from 0.07370 to 0.07208, saving model to best.h5



Epoch 10: finished saving model to best.h5
156/156 ━━━━━━━━━━━━━━━━━━━━ 9s 60ms/step - accuracy: 0.9965 - loss: 0.0376 - val_accuracy: 0.9828 - val_loss: 0.0721 - learning_rate: 2.0000e-04
Epoch 11/12
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9973 - loss: 0.0356
Epoch 11: val_loss did not improve from 0.07208
156/156 ━━━━━━━━━━━━━━━━━━━━ 10s 64ms/step - accuracy: 0.9972 - loss: 0.0352 - val_accuracy: 0.9824 - val_loss: 0.0734 - learning_rate: 2.0000e-04
Epoch 12/12
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9973 - loss: 0.0341
Epoch 12: val_loss did not improve from 0.07208
156/156 ━━━━━━━━━━━━━━━━━━━━ 9s 55ms/step - accuracy: 0.9973 - loss: 0.0340 - val_accuracy: 0.9822 - val_loss: 0.0726 - learning_rate: 2.0000e-04
Restoring model weights from the end of the best epoch: 10.
Epoch 1/8
155/156 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.9624 - loss: 0.1372
Epoch 1: val_loss did not improve from 0.07208
156/156 ━━━━━━━━━━━━━━━━━━━━ 84s 161ms/step - accura

In [ ]:
# 6. Оценка и предсказания

val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"✅ val_loss = {val_loss:.6f}  (< 0.3)")

# Тестовый датасет
def test_gen():
    for fpath in test_files:
        img = tf.io.read_file(fpath)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
        yield img

test_ds = tf.data.Dataset.from_generator(test_gen, output_types=tf.float32)
test_ds = test_ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)

preds = model.predict(test_ds, verbose=1)
preds = np.clip(preds, 1e-6, 1 - 1e-6).flatten()

✅ val_loss = 0.072002  (< 0.3)
98/98 ━━━━━━━━━━━━━━━━━━━━ 98s 843ms/step


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [ ]:
# 7. submission.csv
with open('submission.csv', 'w') as f:
    f.write('id,label\n')
    for fname, p in zip(test_files, preds):
        id_num = re.search(r'(\d+)', os.path.basename(fname)).group(1)
        f.write(f'{id_num},{p:.6f}\n')

print("✅ submission.csv создан (12500 строк). Скачиваю...")
files.download('submission.csv')

✅ submission.csv создан (12500 строк). Скачиваю...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>